In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
import os

In [3]:
df = pd.read_csv('../../data/raw/credit_data.csv')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 594642 entries, 0 to 594641
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   step         594642 non-null  int64  
 1   customer     594642 non-null  str    
 2   age          594642 non-null  str    
 3   gender       594642 non-null  str    
 4   zipcodeori   594642 non-null  str    
 5   merchant     594642 non-null  str    
 6   zipmerchant  594642 non-null  str    
 7   category     594642 non-null  str    
 8   amount       594642 non-null  float64
 9   fraud        594642 non-null  int64  
dtypes: float64(1), int64(2), str(7)
memory usage: 45.4 MB


In [5]:
df.describe()

,step,amount,fraud
count,594642.000000,594642.000000,594642.000000
mean,94.986987,37.890191,0.012108
std,51.053526,111.402916,0.109369
min,0.000000,0.000000,0.000000
25%,52.000000,13.740000,0.000000
50%,97.000000,26.900000,0.000000
75%,139.000000,42.540000,0.000000
max,179.000000,8329.960000,1.000000


In [6]:
df.isna().sum()

step           0
customer       0
age            0
gender         0
zipcodeori     0
merchant       0
zipmerchant    0
category       0
amount         0
fraud          0
dtype: int64

In [7]:
df.duplicated().any()

np.False_

In [8]:
df[df.duplicated()]

,step,customer,age,gender,zipcodeori,merchant,zipmerchant,category,amount,fraud


In [9]:
cat_cols, cont_cols, binary_cols = [], [], []

for col in df.columns:
    if df[col].dtype == "str":
        cat_cols.append(col)
    else:
        if df[col].nunique() > 2:
            cont_cols.append(col)
        elif col != "fraud":
            binary_cols.append(col)


print("Categorical Columns:", cat_cols)

print("Continous Columns:", cont_cols)
print("Binary Columns:", binary_cols)

Categorical Columns: ['customer', 'age', 'gender', 'zipcodeori', 'merchant', 'zipmerchant', 'category']
Continous Columns: ['step', 'amount']
Binary Columns: []


In [10]:
cols_to_drop = []
for col in cat_cols:
   unum = df[col].nunique()

   print(f"Unique numbers of {col}s:", unum)

   if unum == 1:
      cols_to_drop.append(col)

print("\nDropping columns due to constant values:", cols_to_drop)

for col in cols_to_drop:
   cat_cols.remove(col)

df.drop(columns=cols_to_drop, inplace=True)

Unique numbers of customers: 4112
Unique numbers of ages: 8
Unique numbers of genders: 4
Unique numbers of zipcodeoris: 1
Unique numbers of merchants: 50
Unique numbers of zipmerchants: 1
Unique numbers of categorys: 15

Dropping columns due to constant values: ['zipcodeori', 'zipmerchant']


In [31]:
# Just checking if the rolling features are working as expected (NEED TO REVERT THIS SORTING BACK TO ORIGINAL ORDER LATER)
df = df.sort_values(["customer", "step"]).copy()

cust_groups = df.groupby("customer")

df["prev_step"] = cust_groups["step"].shift(1)
df["time_since_last_transaction"] = df["step"] - df["prev_step"]

df["time_since_last_transaction"] = df["time_since_last_transaction"].fillna(-1)
df["prev_step"] = df["prev_step"].fillna(-1)

df["transaction_count"] = cust_groups.cumcount()
df["transaction_count"] = df["transaction_count"].fillna(0)

df["past_amount_sum"] = cust_groups["amount"].cumsum() - df["amount"]
df["past_amount_sum"] = df["past_amount_sum"].fillna(0)

df["avg_amount"] = df["past_amount_sum"] / df["transaction_count"]
df["avg_amount"] = df["avg_amount"].fillna(df["amount"])

df["std_amount"] = cust_groups["amount"].expanding().std().reset_index(level=0, drop=True).shift(1)
df["std_amount"] = df["std_amount"].fillna(0)

df["avg_amount_ratio"] = df["amount"] / df["avg_amount"]
df["log_amount_ratio"] = np.log1p(df["avg_amount_ratio"])

df["amount_zscore"] = (df["amount"] - df["avg_amount"]) / df["std_amount"]
df["amount_zscore"] = df["amount_zscore"].fillna(0)
df["amount_zscore"] = df["amount_zscore"].replace([np.inf, -np.inf], 0)

df["merchant_count"] = df.groupby(["customer", "merchant"]).cumcount()
df["category_count"] = df.groupby(["customer", "category"]).cumcount()

df["prev_merchant_step"] = df.groupby(["customer", "merchant"])["step"].shift(1)
df["time_since_last_merchant_transaction"] = df["step"] - df["prev_merchant_step"]

df["time_since_last_merchant_transaction"] = df["time_since_last_merchant_transaction"].fillna(-1)
df["prev_merchant_step"] = df["prev_merchant_step"].fillna(-1)

df

,step,customer,age,gender,merchant,category,amount,fraud,prev_step,time_since_last_transaction,...,amount_ratio,log_amount_ratio,amount_zscore,merchant_count,category_count,prev_merchant_step,time_since_last_merchant_transaction,past_amount_sum,avg_amount_ratio,avg_amount
80562,30,'C1000148617','5','M','M1888755466','es_otherservices',143.87,0,-1.0,-1.0,...,1.000000,0.693147,0.000000,0,0,-1.0,-1.0,0.00,1.000000,143.870000
105385,38,'C1000148617','5','M','M1741626453','es_sportsandtoys',16.69,0,30.0,8.0,...,0.116008,0.109758,0.000000,0,0,-1.0,-1.0,143.87,0.116008,143.870000
117325,42,'C1000148617','5','M','M1888755466','es_otherservices',56.18,0,38.0,4.0,...,0.699801,0.530511,-0.267987,1,1,30.0,12.0,160.56,0.699801,80.280000
120413,43,'C1000148617','5','M','M840466850','es_tech',14.74,0,42.0,1.0,...,0.204023,0.185669,-0.883434,0,0,-1.0,-1.0,216.74,0.204023,72.246667
124901,44,'C1000148617','5','M','M1823072687','es_transportation',47.42,0,43.0,1.0,...,0.819423,0.598519,-0.172931,0,0,-1.0,-1.0,231.48,0.819423,57.870000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
573564,174,'C999723254','2','M','M1823072687','es_transportation',31.94,0,173.0,1.0,...,1.097459,0.740727,0.079722,73,100,173.0,1.0,3405.12,1.097459,29.103590
581160,176,'C999723254','2','M','M1823072687','es_transportation',1.92,0,174.0,2.0,...,0.065917,0.063835,-0.767987,74,101,174.0,2.0,3437.06,0.065917,29.127627
585495,177,'C999723254','2','M','M85975013','es_food',62.55,0,176.0,1.0,...,2.164435,1.151975,0.951539,7,7,170.0,7.0,3438.98,2.164435,28.898992
587473,178,'C999723254','2','M','M1823072687','es_transportation',25.96,0,177.0,1.0,...,0.889668,0.636401,-0.091074,75,102,176.0,2.0,3501.53,0.889668,29.179417


In [32]:
df = df.sort_index()
df

,step,customer,age,gender,merchant,category,amount,fraud,prev_step,time_since_last_transaction,...,amount_ratio,log_amount_ratio,amount_zscore,merchant_count,category_count,prev_merchant_step,time_since_last_merchant_transaction,past_amount_sum,avg_amount_ratio,avg_amount
0,0,'C352968107','2','M','M348934600','es_transportation',39.68,0,-1.0,-1.0,...,1.284028,0.693147,0.000000,0,0,-1.0,-1.0,0.00,1.000000,39.680000
1,0,'C2054744914','4','F','M1823072687','es_transportation',26.89,0,-1.0,-1.0,...,0.780126,0.693147,0.000000,0,0,-1.0,-1.0,0.00,1.000000,26.890000
2,0,'C1760612790','3','M','M348934600','es_transportation',17.25,0,-1.0,-1.0,...,0.395733,0.693147,0.000000,0,0,-1.0,-1.0,0.00,1.000000,17.250000
3,0,'C757503768','5','M','M348934600','es_transportation',35.72,0,-1.0,-1.0,...,1.103832,0.693147,0.000000,0,0,-1.0,-1.0,0.00,1.000000,35.720000
4,0,'C1315400589','3','F','M348934600','es_transportation',25.81,0,-1.0,-1.0,...,0.862958,0.693147,0.000000,0,0,-1.0,-1.0,0.00,1.000000,25.810000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594637,179,'C1753498738','3','F','M1823072687','es_transportation',20.53,0,178.0,1.0,...,0.621079,0.483092,-0.372043,154,163,178.0,1.0,6049.13,0.621079,33.055355
594638,179,'C650108285','4','F','M1823072687','es_transportation',50.73,0,178.0,1.0,...,1.448606,0.895519,0.535767,109,144,178.0,1.0,5988.40,1.448606,35.019883
594639,179,'C123623130','2','F','M349281107','es_fashion',22.44,0,178.0,1.0,...,0.716318,0.540181,-0.364592,0,0,-1.0,-1.0,5231.59,0.716318,31.326886
594640,179,'C1499363341','5','M','M1823072687','es_transportation',14.46,0,178.0,1.0,...,0.514853,0.415319,-0.723707,96,147,178.0,1.0,4606.05,0.514853,28.085671


In [ ]:
first_txns = df[df["transaction_count"] == 0]

print((first_txns["avg_amount"] == first_txns["amount"]).all())
print((first_txns["avg_amount_ratio"] == 1.0).all())
print((first_txns["amount_zscore"] == 0).all()A
print((first_txns["prev_step"] == -1).all()

True
True
True
True
